In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelArchitecture:
    model_architecture: str
    library: str
    model_name: str
    encoder: str
    encoder_weights: str
    classes: int
    activation: str

@dataclass(frozen=True)
class PrepareBaseModelConfig:
    root_dir: Path
    base_model_path: Path
    modelArchitecture: ModelArchitecture

In [3]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_prepare_base_model_config(self) -> PrepareBaseModelConfig:
        config = self.config.prepare_base_model_config
        params = self.params.prepare_base_model_params
        
        model_dir = os.path.join(config.root_dir, f"{params.model_name}_{params.encoder}")
        create_directories([config.root_dir, model_dir])
        
        modelArchitecture = ModelArchitecture(
            model_architecture = params.model_architecture,
            library = params.library,
            model_name = params.model_name,
            encoder = params.encoder,
            encoder_weights = params.encoder_weights,
            classes = params.classes,
            activation = params.activation,
        )

        prepare_base_model_config = PrepareBaseModelConfig(
            root_dir=Path(config.root_dir),
            base_model_path=Path(os.path.join(model_dir, "base_model.pth")),
            modelArchitecture=modelArchitecture
        )

        return prepare_base_model_config

In [4]:
import sys, torch
import segmentation_models_pytorch as smp

from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException

from pneumonia_segmentation.utils.model_utils import build_model

MODEL_MAP = {
    "unet": smp.Unet,
    "unetpp": smp.UnetPlusPlus,
    "deeplabv3": smp.DeepLabV3,
    "manet": smp.MAnet
}

class PrepareBaseModel:
    def __init__(self, config: PrepareBaseModelConfig):
        self.config = config
        self.activation = None if self.config.modelArchitecture.activation == "None" else self.config.modelArchitecture.activation
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
    
    def _build_model(self):
        model_name = self.config.modelArchitecture.model_name
        if model_name.lower() not in MODEL_MAP:
            raise ValueError(f"Model {model_name} is not supported. Choose from {list(MODEL_MAP.keys())}")
    
        return MODEL_MAP[model_name.lower()](
            encoder_name    = self.config.modelArchitecture.encoder,
            encoder_weights = self.config.modelArchitecture.encoder_weights,
            classes         = self.config.modelArchitecture.classes,
            activation      = self.activation
        ) 

    def main(self):
        base_model = self._build_model().to(self.device) 
        torch.save(base_model, self.config.base_model_path)
        logging.info(f"Base model saved at {self.config.base_model_path}")

d:\Deep_Learning_Object_Detection\MLOPs\pneumonia-segmentation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
try:
    config = ConfigurationManger()
    prepare_base_model_config = config.get_prepare_base_model_config()
    prepare_base_model = PrepareBaseModel(config=prepare_base_model_config)
    prepare_base_model.main()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-05 19:53:36,883: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-05 19:53:36,888: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-05 19:53:36,896: INFO: common: created directory at: artifacts]
[2026-04-05 19:53:36,904: INFO: common: created directory at: artifacts/prepare_base_model]
[2026-04-05 19:53:36,912: INFO: common: created directory at: artifacts/prepare_base_model\unetpp_efficientnet-b3]
[2026-04-05 19:53:38,727: INFO: 752879436: Base model saved at artifacts\prepare_base_model\unetpp_efficientnet-b3\base_model.pth]
